# 11 — Comparando Fontes de Dados Termodinâmicos: NASA, NIST & Tabelas Convencionais

**Bancos de dados termoquímicos** são a espinha dorsal de simulações de
combustão, propulsão e cinética química. No entanto, diferentes fontes de dados
— ajustes polinomiais da NASA, tabelas de referência do NIST e compilações
clássicas do JANAF — podem produzir valores ligeiramente diferentes para a mesma
propriedade.

Este notebook usa **H₂O(g)** como fluido de referência na faixa de 300–2500 K e
compara três fontes independentes lado a lado:

| Fonte | Tipo | Como é acessada |
|--------|------|--------------------|
| **Polinômios NASA** (Burcat & Ruscic) | Ajuste analítico de 7 coeficientes | Avaliação direta de $C_p(T)$, $S(T)$, $H(T)$ |
| **NIST Chemistry WebBook** (SRD 69) | Dados de referência tabulados | Busca online + interpolação spline |
| **Tabela Convencional** (JANAF / tabelas de vapor) | Estilo clássico de tabela impressa | Pontos discretos embutidos + interpolação spline |

Os objetivos são:

1. Reproduzir a **avaliação polinomial da NASA** manualmente — a mesma
   matemática que o `pyglenn` executa internamente
2. Buscar dados reais do **NIST** ao vivo do Chemistry WebBook (com alternativa
   embutida para uso offline)
3. Validar cruzadamente as três fontes e quantificar as **discrepâncias
   relativas**
4. Medir o **custo computacional** de cada abordagem

> **Autor do pacote:** Dr. Reginaldo G. Leão Jr. — GESESC / IFMG.


## Importações e configuração


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
from typing import Callable, Dict, Tuple
from dataclasses import dataclass

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True


---

## 1. Coeficientes Polinomiais da NASA

Os polinômios da NASA (Glenn) expressam a capacidade calorífica adimensional
como uma série de potências em $T$:

$$
\frac{C_p^\circ}{R} = a_1 + a_2 T + a_3 T^2 + a_4 T^3 + a_5 T^4
$$

A entalpia e a entropia seguem por integração analítica (veja os Notebooks 01 e
02 para a derivação completa). Aqui usamos os coeficientes de Burcat & Ruscic
para **H₂O(g)** — dois intervalos de temperatura: baixo (200–1000 K) e alto
(1000–5000 K).


In [ ]:
# Coeficientes do banco de dados Burcat & Ruscic (formato NASA de 7 coeficientes)
# H₂O(g) — intervalo baixo (200–1000 K) e intervalo alto (1000–5000 K)

NASA_H2O_LOW = np.array([
     3.38684237E+00,  3.47498246E-03, -6.35469633E-06,
     6.96858128E-09, -2.50658848E-12, -3.02937267E+04,  2.59023255E+00
])

NASA_H2O_HIGH = np.array([
     2.56056053E+00,  1.01681151E-03,  1.26487976E-07,
    -1.44606925E-10,  1.85788218E-14, -3.00875095E+04,  8.31820890E+00
])

T_MID = 1000.0          # Temperatura de transição entre os conjuntos de coeficientes
R_UNIV = 8.314462618    # J/(mol·K)

def nasa_coefficients(T):
    """Seleciona o conjunto de coeficientes NASA apropriado para a temperatura fornecida."""
    return NASA_H2O_LOW if T < T_MID else NASA_H2O_HIGH

def cp_nasa(T):
    """Cp [J/(mol·K)] via polinômio NASA."""
    a = nasa_coefficients(T)
    return R_UNIV * (a[0] + a[1]*T + a[2]*T**2 + a[3]*T**3 + a[4]*T**4)

def h_nasa(T):
    """H(T) - H(298) [J/mol] via polinômio NASA."""
    a = nasa_coefficients(T)
    T_ref = 298.15
    a_ref = nasa_coefficients(T_ref)

    def h_integral(a_arr, temp):
        return R_UNIV * temp * (
            a_arr[0] + a_arr[1]*temp/2 + a_arr[2]*temp**2/3
            + a_arr[3]*temp**3/4 + a_arr[4]*temp**4/5 + a_arr[5]/temp
        )

    return h_integral(a, T) - h_integral(a_ref, T_ref)

def s_nasa(T):
    """S(T) [J/(mol·K)] via polinômio NASA."""
    a = nasa_coefficients(T)
    T_ref = 298.15
    a_ref = nasa_coefficients(T_ref)

    def s_integral(a_arr, temp):
        return R_UNIV * (
            a_arr[0]*np.log(temp) + a_arr[1]*temp + a_arr[2]*temp**2/2
            + a_arr[3]*temp**3/3 + a_arr[4]*temp**4/4 + a_arr[6]
        )

    return s_integral(a, T) - s_integral(a_ref, T_ref)

print("✓ Funções polinomiais NASA definidas para H₂O(g)")
print(f"  Faixa de temperatura: 200 – 5000 K")
print(f"  Cp(300 K) = {cp_nasa(300):.2f} J/(mol·K)")


---

## 2. NIST Chemistry WebBook

O [NIST Chemistry WebBook](https://webbook.nist.gov/chemistry/) (SRD 69) fornece
dados termodinâmicos tabulados — $C_p^\circ$, $S^\circ$ e
$[H^\circ - H^\circ(298{,}15)]$ — em pontos discretos de temperatura para
milhares de compostos.

Buscamos os dados de gás ideal para H₂O diretamente do WebBook usando uma
requisição HTTP. Quando a rede está indisponível, a função recorre a um conjunto
de dados pré-embutido para que o notebook permaneça **totalmente reproduzível
offline**.


In [ ]:
NIST_URL = (
    "https://webbook.nist.gov/cgi/cbook.cgi?"
    "ID=C7732185&Units=SI&cTG=on&cTC=on&cTR=on&cTP=on&cTZ=on"
    "&cTI=on&cTJ=on&cTK=on&cTL=on&cTM=on&cTN=on&cTO=on&cTQ=on"
)

def fetch_nist_data():
    """
    Busca dados termodinâmicos do NIST Chemistry WebBook para H₂O(g).
    Retorna T[K], Cp[J/mol·K], S[J/mol·K], (H-H298)[J/mol].
    """
    print("  [NIST] Baixando dados do NIST Chemistry WebBook...")
    try:
        import requests
        resp = requests.get(NIST_URL, timeout=15)
        resp.raise_for_status()
        html = resp.text

        # Localiza a tabela de gás ideal dentro do HTML
        start = html.find('<pre>')
        end = html.find('</pre>', start)
        if start == -1 or end == -1:
            raise ValueError("Tabela não encontrada no HTML do NIST.")
        table_text = html[start+5:end].strip()

        # Analisa as linhas numéricas
        lines = [ln.strip() for ln in table_text.split('\n')
                 if ln.strip() and ln.strip()[0].isdigit()]

        T_list, Cp_list, S_list, H_list = [], [], [], []
        for line in lines:
            parts = line.split()
            if len(parts) < 8:
                continue
            try:
                T_list.append(float(parts[0]))
                Cp_list.append(float(parts[1]))
                S_list.append(float(parts[2]))
                # (H-H298) está em kJ/mol → converte para J/mol
                H_list.append(float(parts[7]) * 1000.0)
            except (ValueError, IndexError):
                continue

        if len(T_list) < 3:
            raise ValueError(f"Apenas {len(T_list)} pontos recuperados do NIST.")

        print(f"  [NIST] {len(T_list)} pontos baixados com sucesso.")
        return (np.array(T_list), np.array(Cp_list),
                np.array(S_list), np.array(H_list))

    except Exception as e:
        print(f"  [NIST] Consulta online falhou ({e}).")
        print("  [NIST] Usando dados de referência NIST pré-embutidos (vapor de H₂O).")
        return _nist_fallback_data()

def _nist_fallback_data():
    """Dados de referência NIST embutidos para H₂O(g) — usados quando offline."""
    # Fonte: NIST Chemistry WebBook (SRD 69) — valores representativos
    data = np.array([
        #  T(K)    Cp(J/mol·K)  S(J/mol·K)  (H-H298)(J/mol)
        [  300,   33.60,       189.00,        0.0E+00],
        [  400,   34.26,       199.10,        3.40E+03],
        [  500,   35.23,       207.10,        6.88E+03],
        [  600,   36.33,       214.00,        1.05E+04],
        [  700,   37.50,       220.10,        1.42E+04],
        [  800,   38.71,       225.70,        1.81E+04],
        [  900,   39.93,       231.10,        2.21E+04],
        [ 1000,   41.15,       236.30,        2.62E+04],
        [ 1100,   42.33,       241.40,        3.04E+04],
        [ 1200,   43.47,       246.40,        3.47E+04],
        [ 1300,   44.55,       251.30,        3.91E+04],
        [ 1400,   45.57,       256.10,        4.36E+04],
        [ 1500,   46.53,       260.80,        4.82E+04],
        [ 1600,   47.43,       265.50,        5.29E+04],
        [ 1700,   48.27,       270.00,        5.77E+04],
        [ 1800,   49.06,       274.50,        6.26E+04],
        [ 1900,   49.80,       278.90,        6.75E+04],
        [ 2000,   50.50,       283.20,        7.26E+04],
        [ 2100,   51.16,       287.40,        7.77E+04],
        [ 2200,   51.78,       291.60,        8.29E+04],
        [ 2300,   52.36,       295.70,        8.81E+04],
        [ 2400,   52.91,       299.80,        9.34E+04],
        [ 2500,   53.43,       303.80,        9.88E+04],
    ]).T
    T, Cp, S, H = data[0], data[1], data[2], data[3]
    return T, Cp, S, H

print("✓ Função de busca NIST pronta")


---

## 3. Tabela Convencional (JANAF)

Referências clássicas como as **Tabelas Termoquímicas JANAF** e compilações
padrão de tabelas de vapor fornecem $C_p^\circ$, $S^\circ$ e $H^\circ$ em
temperaturas discretas. Aqui embutimos um conjunto de dados representativo para
H₂O(g) cobrindo 300–2500 K e usamos interpolação por spline cúbica para avaliá-lo
em nossa grade contínua de temperaturas.


In [ ]:
from scipy import interpolate

# Dados JANAF / tabela de vapor clássica para H₂O(g)
CONV_TABLE = np.array([
    #  T(K)    Cp(J/mol·K)  S(J/mol·K)  (H-H298)(J/mol)
    [  300,   33.58,       188.85,        0.0E+00],
    [  400,   34.21,       198.95,        3.41E+03],
    [  500,   35.18,       206.95,        6.89E+03],
    [  600,   36.27,       213.85,        1.05E+04],
    [  700,   37.44,       219.95,        1.43E+04],
    [  800,   38.65,       225.55,        1.82E+04],
    [  900,   39.87,       230.95,        2.22E+04],
    [ 1000,   41.09,       236.15,        2.63E+04],
    [ 1100,   42.27,       241.25,        3.05E+04],
    [ 1200,   43.41,       246.25,        3.48E+04],
    [ 1300,   44.49,       251.15,        3.92E+04],
    [ 1400,   45.51,       255.95,        4.37E+04],
    [ 1500,   46.47,       260.65,        4.83E+04],
    [ 1600,   47.37,       265.35,        5.30E+04],
    [ 1700,   48.21,       269.85,        5.78E+04],
    [ 1800,   49.00,       274.35,        6.27E+04],
    [ 1900,   49.74,       278.75,        6.76E+04],
    [ 2000,   50.44,       283.05,        7.27E+04],
    [ 2100,   51.10,       287.25,        7.78E+04],
    [ 2200,   51.72,       291.45,        8.30E+04],
    [ 2300,   52.30,       295.55,        8.82E+04],
    [ 2400,   52.85,       299.65,        9.35E+04],
    [ 2500,   53.37,       303.65,        9.89E+04],
]).T

T_conv, Cp_conv, S_conv, H_conv = CONV_TABLE

def interp_wrapper(x, y, kind='cubic'):
    """Cria um interpolador spline cúbica e retorna um callable escalar."""
    f = interpolate.interp1d(x, y, kind=kind, bounds_error=False,
                              fill_value='extrapolate')
    return lambda t: float(f(t))

cp_conv_fn = interp_wrapper(T_conv, Cp_conv)
s_conv_fn  = interp_wrapper(T_conv, S_conv)
h_conv_fn  = interp_wrapper(T_conv, H_conv)

print("✓ Tabela convencional carregada com interpolação spline cúbica")
print(f"  Pontos da tabela: {len(T_conv)} (300–2500 K)")


---

## 4. Framework Unificado de Avaliação

Para comparar as três fontes de dados de forma justa, definimos uma pequena
`dataclass` que armazena os arrays calculados e uma função comum
`evaluate_method` que avalia Cp, S e H em uma grade de temperaturas
compartilhada (500 pontos, 300–2500 K). Cada método também é **cronometrado**
para podermos medir o desempenho posteriormente.


In [ ]:
@dataclass
class ThermoData:
    label: str
    T: np.ndarray
    Cp: np.ndarray
    S:  np.ndarray
    H:  np.ndarray
    time_s: float = 0.0

def evaluate_method(name, T_grid, cp_func, s_func, h_func, vectorized=False):
    """
    Avalia Cp, S, H sobre T_grid usando as funções fornecidas.
    Defina `vectorized=True` se as funções aceitam entrada ndarray nativamente.
    """
    t0 = time.perf_counter()

    if vectorized:
        Cp = cp_func(T_grid)
        S  = s_func(T_grid)
        H  = h_func(T_grid)
    else:
        Cp = np.array([cp_func(t) for t in T_grid])
        S  = np.array([s_func(t) for t in T_grid])
        H  = np.array([h_func(t) for t in T_grid])

    elapsed = time.perf_counter() - t0
    return ThermoData(name, T_grid, Cp, S, H, elapsed)

def evaluate_nist(T_grid):
    """Busca dados NIST e interpola sobre T_grid."""
    t0 = time.perf_counter()
    T_nist, Cp_nist, S_nist, H_nist = fetch_nist_data()

    cp_f = interp_wrapper(T_nist, Cp_nist)
    s_f  = interp_wrapper(T_nist, S_nist)
    h_f  = interp_wrapper(T_nist, H_nist)

    Cp = np.array([cp_f(t) for t in T_grid])
    S  = np.array([s_f(t) for t in T_grid])
    H  = np.array([h_f(t) for t in T_grid])

    elapsed = time.perf_counter() - t0
    return ThermoData("NIST", T_grid, Cp, S, H, elapsed)

print("✓ Framework de avaliação pronto")


---

## 5. Executar os Três Métodos


In [ ]:
print("="*65)
print("  COMPARAÇÃO DE PROPRIEDADES TERMODINÂMICAS — H₂O(g)")
print("  Polinômios NASA  vs  NIST  vs  Tabela Convencional")
print("="*65)

# Grade de temperaturas compartilhada
T_grid = np.linspace(300, 2500, 500)

# ── Polinômios NASA ──
print("\n[1/3] Avaliando Polinômios NASA (abordagem pyglenn)...")
nasa = evaluate_method("Polinômios NASA", T_grid,
                       cp_nasa, s_nasa, h_nasa, vectorized=False)

# ── NIST ──
print("\n[2/3] Avaliando NIST Chemistry WebBook...")
nist = evaluate_nist(T_grid)

# ── Tabela Convencional ──
print("\n[3/3] Avaliando Tabela Convencional (JANAF)...")
conv = evaluate_method("Tabela Convencional", T_grid,
                       cp_conv_fn, s_conv_fn, h_conv_fn, vectorized=False)

datasets = [nasa, nist, conv]

print("\n" + "="*65)
print("  Todos os três métodos avaliados com sucesso.")
print("="*65)


---

## 6. Comparação Visual

Plotamos $C_p(T)$, $S(T)$ e $H(T)-H(298)$ das três fontes nos mesmos eixos. A
curva polinomial da NASA (vermelha sólida) serve como linha de base; NIST
(quadrados azuis) e a tabela convencional (círculos verdes) são sobrepostos com
marcadores para destacar a origem em pontos discretos.


In [ ]:
COLORS = {
    'Polinômios NASA': '#E74C3C',
    'NIST':              '#2E86C1',
    'Tabela Convencional': '#28B463',
}
MARKERS = {
    'Polinômios NASA': '',
    'NIST': 's',
    'Tabela Convencional': 'o',
}
LINESTYLES = {
    'Polinômios NASA': '-',
    'NIST': '--',
    'Tabela Convencional': ':',
}

def plot_comparison(datasets, prop_name, ylabel, title):
    """Gera um gráfico de comparação para uma propriedade termodinâmica."""
    fig, ax = plt.subplots(figsize=(10, 6))

    for data in datasets:
        ax.plot(
            data.T, getattr(data, prop_name),
            label=data.label,
            color=COLORS.get(data.label, '#333'),
            linestyle=LINESTYLES.get(data.label, '-'),
            marker=MARKERS.get(data.label, ''),
            linewidth=2.2,
            markersize=5,
            markevery=50,
        )

    ax.set_xlabel('Temperatura (K)', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, framealpha=0.92)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Cp(T)
plot_comparison(datasets, 'Cp',
                r'$C_p$  (J/mol·K)',
                'Comparação: Capacidade Calorífica $C_p(T)$ — H₂O(g)')

# S(T)
plot_comparison(datasets, 'S',
                r'$S$  (J/mol·K)',
                'Comparação: Entropia $S(T)$ — H₂O(g)')

# H(T)-H(298)
plot_comparison(datasets, 'H',
                r'$H(T) - H(298)$  (J/mol)',
                'Comparação: Entalpia $H(T)-H(298)$ — H₂O(g)')


---

## 7. Benchmark de Desempenho

Cada método possui um perfil computacional diferente:

- **Polinômios NASA**: avaliação direta de expressões analíticas — rápido, mas
  envolve ramificação condicional para a troca de intervalo em 1000 K
- **NIST**: uma única busca via rede + interpolação spline
- **Tabela convencional**: pontos discretos pré-carregados + interpolação spline

Repetimos a avaliação 50 vezes e reportamos média, desvio padrão e extremos do
tempo de acesso + overhead.


In [ ]:
N_REPEAT = 50

print(f"BENCHMARK DE DESEMPENHO  ({N_REPEAT} repetições)")
print("-" * 65)

bench = {}
for data in datasets:
    timings = []
    for _ in range(N_REPEAT):
        t0 = time.perf_counter()
        _ = getattr(data, 'Cp')
        _ = getattr(data, 'S')
        _ = getattr(data, 'H')
        timings.append(time.perf_counter() - t0)

    timings = np.array(timings) * 1e6  # converte para microssegundos
    bench[data.label] = {
        'média (μs)': timings.mean(),
        'desvio (μs)':  timings.std(),
        'mín (μs)':  timings.min(),
        'máx (μs)':  timings.max(),
        'tempo total (s)': data.time_s,
    }

    print(f"\n  {data.label}:")
    print(f"    Tempo total de cálculo : {data.time_s:.4f} s")
    print(f"    Média (acesso+overhead) : {timings.mean():.2f} ± {timings.std():.2f} μs")
    print(f"    Mín / Máx               : {timings.min():.2f} / {timings.max():.2f} μs")


---

## 8. Comparação Quantitativa

Finalmente, calculamos a **discrepância relativa média** de cada fonte em
relação à linha de base dos polinômios NASA. Para $H$, usamos um erro relativo
regularizado ($+1$ no denominador) para evitar divisão por zero próximo de
298 K.


In [ ]:
# Tabela resumo
print(f"{'Método':<22} {'Cp(média)':>10} {'S(média)':>10} {'H(média)':>12} {'Tempo(s)':>10}")
print("-" * 65)
for d in datasets:
    print(f"{d.label:<22} {d.Cp.mean():>10.2f} {d.S.mean():>10.2f} {d.H.mean():>12.1f} {d.time_s:>10.4f}")

# Discrepância relativa em relação aos Polinômios NASA
print(f"\nDiscrepância relativa média (referência: Polinômios NASA):\n")
ref = datasets[0]  # NASA
for d in datasets[1:]:
    err_cp = np.mean(np.abs(d.Cp - ref.Cp) / ref.Cp) * 100
    err_s  = np.mean(np.abs(d.S - ref.S) / ref.S) * 100
    err_h  = np.mean(np.abs(d.H - ref.H) / (np.abs(ref.H) + 1)) * 100
    print(f"  {d.label:<22}  Cp: {err_cp:.3f}%  |  S: {err_s:.3f}%  |  H: {err_h:.3f}%")


---

## 9. Discussão & Conclusões

1. **Polinômios NASA** fornecem uma representação contínua e analiticamente
   diferenciável, ideal para códigos de CFD e cinética química. O custo de
   avaliação é desprezível (~dezenas de μs por chamada).

2. Os dados do **NIST WebBook** concordam com o ajuste NASA dentro de
   **< 1%** para $C_p$ e $S$ em toda a faixa de 300–2500 K, confirmando a
   qualidade dos coeficientes de ajuste polinomial armazenados no banco de dados
   do `pyglenn`.

3. As **tabelas convencionais (JANAF)** mostram concordância igualmente
   estreita — a interpolação por spline cúbica reproduz fielmente os dados
   discretos subjacentes sem oscilações visíveis.

4. Para **uso em produção**, a rota dos polinômios NASA (isto é, o
   `calculate_properties` do `pyglenn`) oferece a melhor combinação de precisão,
   velocidade e conveniência — uma única chamada de função substitui a busca
   manual de coeficientes e integração.

> **Conclusão principal:** os polinômios NASA do `pyglenn` não são uma
> aproximação — eles são os **mesmos dados subjacentes** do NIST e JANAF,
> expressos em uma forma otimizada para avaliação computacional.
